[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/mlops/04-pipelines-automatizados.ipynb)

# Pipelines Automatizados para LLMs

> Aprende a construir pipelines de procesamiento automático con LLMs:
> ingesta, enriquecimiento, evaluación y publicación de resultados.

## Objetivos
- Construir un pipeline ETL enriquecido con LLM
- Implementar reintentos y manejo de errores robusto
- Procesar documentos en batch con control de concurrencia
- Orquestar pasos de pipeline con dependencias

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas --quiet

## 2. Arquitectura del pipeline

In [ ]:
import anthropic
import json
import time
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, Callable
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("""=== ARQUITECTURA DE PIPELINE LLM ===

INGESTA → PREPROCESADO → ENRIQUECIMIENTO LLM → VALIDACIÓN → ALMACENAMIENTO

Pasos del pipeline de ejemplo (clasificación de noticias):
  1. INGESTA: cargar artículos desde JSON/CSV
  2. LIMPIEZA: normalizar texto, eliminar duplicados
  3. CLASIFICACIÓN: Claude clasifica tema y sentimiento
  4. EXTRACCIÓN: entidades, palabras clave, resumen
  5. VALIDACIÓN: verificar schema y calidad
  6. EXPORTACIÓN: JSON/CSV enriquecido

Patrones de resiliencia:
  - Reintentos con backoff exponencial
  - Dead letter queue para fallos permanentes
  - Checkpointing para reanudación
  - Rate limiting para respetar quotas de API
""")

## 3. Implementar el pipeline con reintentos

In [ ]:
@dataclass
class ResultadoPaso:
    """Resultado de un paso del pipeline."""
    exito: bool
    datos: dict = field(default_factory=dict)
    error: Optional[str] = None
    intentos: int = 1
    latencia_s: float = 0.0

def con_reintentos(func: Callable, max_intentos: int = 3, backoff: float = 1.0):
    """Decorador que aplica reintentos con backoff exponencial."""
    def wrapper(*args, **kwargs):
        ultimo_error = None
        for intento in range(1, max_intentos + 1):
            try:
                return func(*args, **kwargs), intento
            except Exception as e:
                ultimo_error = str(e)
                if intento < max_intentos:
                    espera = backoff * (2 ** (intento - 1))
                    time.sleep(espera)
        return None, max_intentos
    return wrapper

def paso_clasificar(articulo: dict) -> ResultadoPaso:
    """Paso 1: clasificar tema y sentimiento del artículo."""
    prompt = f"""Analiza este artículo de noticias.

Título: {articulo['titulo']}
Texto: {articulo['texto'][:500]}

Responde SOLO con JSON:
{{"tema": "tecnología|economía|política|ciencia|cultura|deportes|otro",
  "sentimiento": "positivo|negativo|neutro",
  "urgencia": "alta|media|baja"}}"""

    inicio = time.perf_counter()
    resp = client.messages.create(
        model=MODELO, max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    latencia = time.perf_counter() - inicio

    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    datos = json.loads(resp)
    return ResultadoPaso(exito=True, datos=datos, latencia_s=latencia)

def paso_extraer(articulo: dict) -> ResultadoPaso:
    """Paso 2: extraer entidades y generar resumen."""
    prompt = f"""Extrae información clave de este artículo.

Título: {articulo['titulo']}
Texto: {articulo['texto'][:500]}

Responde SOLO con JSON:
{{"resumen": "<1 frase>",
  "personas": ["nombre1"],
  "organizaciones": ["org1"],
  "palabras_clave": ["kw1", "kw2", "kw3"]}}"""

    inicio = time.perf_counter()
    resp = client.messages.create(
        model=MODELO, max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    latencia = time.perf_counter() - inicio

    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    datos = json.loads(resp)
    return ResultadoPaso(exito=True, datos=datos, latencia_s=latencia)

# Dataset de artículos
ARTICULOS = [
    {"id": "art_001", "titulo": "OpenAI lanza nuevo modelo GPT-5 con capacidades multimodales",
     "texto": "OpenAI ha anunciado el lanzamiento de GPT-5, su nuevo modelo de lenguaje que incluye capacidades avanzadas de visión, audio y generación de código. El modelo supera en benchmarks clave a sus predecesores."},
    {"id": "art_002", "titulo": "El Banco Central sube los tipos de interés al 4.5%",
     "texto": "En una decisión que sorprendió a los mercados, el banco central europeo ha elevado los tipos de interés en 25 puntos básicos, citando la persistente inflación como principal causa."},
    {"id": "art_003", "titulo": "Nuevos hallazgos sobre la materia oscura en galaxias distantes",
     "texto": "Un equipo internacional de astrofísicos ha publicado evidencias que sugieren una nueva comprensión de la distribución de materia oscura en galaxias espirales, poniendo en cuestión modelos establecidos."},
    {"id": "art_004", "titulo": "Escándalo político sacude al gobierno tras filtración de emails",
     "texto": "Una filtración masiva de correos electrónicos internos ha revelado conversaciones comprometedoras entre altos funcionarios del gobierno, generando una crisis de confianza."},
]

print(f"Pipeline configurado con {len(ARTICULOS)} artículos")

## 4. Ejecutar pipeline con concurrencia controlada

In [ ]:
def procesar_articulo(articulo: dict) -> dict:
    """Procesa un artículo completo a través de todos los pasos."""
    resultado = {"id": articulo["id"], "titulo": articulo["titulo"], "pasos": {}}

    # Paso 1: clasificación
    try:
        r1 = paso_clasificar(articulo)
        resultado["pasos"]["clasificacion"] = {"exito": True, "latencia": r1.latencia_s}
        resultado.update(r1.datos)
    except Exception as e:
        resultado["pasos"]["clasificacion"] = {"exito": False, "error": str(e)}
        resultado.update({"tema": "desconocido", "sentimiento": "neutro", "urgencia": "baja"})

    # Paso 2: extracción
    try:
        r2 = paso_extraer(articulo)
        resultado["pasos"]["extraccion"] = {"exito": True, "latencia": r2.latencia_s}
        resultado.update(r2.datos)
    except Exception as e:
        resultado["pasos"]["extraccion"] = {"exito": False, "error": str(e)}
        resultado.update({"resumen": "", "personas": [], "organizaciones": [], "palabras_clave": []})

    resultado["procesado_en"] = datetime.now().isoformat()
    return resultado

print("=== EJECUTANDO PIPELINE ===")
inicio_total = time.perf_counter()
resultados_pipeline = []

# Procesar con concurrencia controlada (máx 2 workers para respetar rate limits)
with ThreadPoolExecutor(max_workers=2) as executor:
    futuros = {executor.submit(procesar_articulo, art): art["id"] for art in ARTICULOS}
    for futuro in as_completed(futuros):
        art_id = futuros[futuro]
        try:
            resultado = futuro.result()
            resultados_pipeline.append(resultado)
            exitos = sum(v["exito"] for v in resultado["pasos"].values())
            total_pasos = len(resultado["pasos"])
            print(f"  ✓ {art_id}: {resultado.get('tema', '?')} | {resultado.get('sentimiento', '?')} | {exitos}/{total_pasos} pasos OK")
        except Exception as e:
            print(f"  ✗ {art_id}: Error - {e}")

tiempo_total = time.perf_counter() - inicio_total
print(f"\nTiempo total: {tiempo_total:.1f}s para {len(ARTICULOS)} artículos")
print(f"Throughput: {len(ARTICULOS)/tiempo_total:.2f} artículos/segundo")

## 5. Exportar resultados y resumen del pipeline

In [ ]:
# Exportar resultados enriquecidos
filas_export = []
for r in resultados_pipeline:
    filas_export.append({
        "id": r["id"],
        "titulo": r["titulo"][:60],
        "tema": r.get("tema", ""),
        "sentimiento": r.get("sentimiento", ""),
        "urgencia": r.get("urgencia", ""),
        "resumen": r.get("resumen", ""),
        "palabras_clave": ", ".join(r.get("palabras_clave", [])),
        "personas": ", ".join(r.get("personas", [])),
        "procesado_en": r.get("procesado_en", ""),
    })

df_resultados = pd.DataFrame(filas_export)
df_resultados.to_csv("pipeline_resultado.csv", index=False, encoding="utf-8")

# Métricas del pipeline
pasos_ok = sum(
    sum(1 for p in r["pasos"].values() if p["exito"])
    for r in resultados_pipeline
)
pasos_total = sum(len(r["pasos"]) for r in resultados_pipeline)

print("=== RESUMEN DEL PIPELINE ===")
print(f"  Artículos procesados: {len(resultados_pipeline)}/{len(ARTICULOS)}")
print(f"  Pasos completados: {pasos_ok}/{pasos_total} ({pasos_ok/pasos_total:.0%})")
print(f"  Distribución de temas: {df_resultados['tema'].value_counts().to_dict()}")
print(f"  Distribución de sentimiento: {df_resultados['sentimiento'].value_counts().to_dict()}")
print(f"  Resultado exportado: pipeline_resultado.csv")
print("\n=== TABLA DE RESULTADOS ===")
print(df_resultados[["id", "tema", "sentimiento", "urgencia", "resumen"]].to_string(index=False))

print("""
=== PRÓXIMOS PASOS PARA PRODUCCIÓN ===
  1. Añadir Apache Airflow o Prefect para orquestación
  2. Usar colas (Redis/RabbitMQ) para desacoplar producción/consumo
  3. Configurar alertas en Grafana/Datadog
  4. Implementar checkpointing en base de datos
  5. Añadir métricas de costes por artículo procesado
""")